### Averaging out replicates

In [ ]:
import os
import re
import pandas as pd
from tqdm import tqdm 

def extract_base(col_name):
    """
    Extracts the base condition name from a column name by removing a replicate marker.
    Recognizes patterns like:
      - rep1, rep 1, rep_1, replicate 1, replicate_1, Replicate, etc.
    even when followed by extra text.
    """
    pattern = re.compile(r"^(.*?)\s*(rep(?:licate)?\s*[_-]?\s*\d+)(.*)$", re.IGNORECASE)
    m = pattern.match(col_name)
    if m:
        # Concatenate the part before the replicate marker with the part after it.
        base = (m.group(1) + m.group(3)).strip()
        if base:
            return base
    return col_name

def format_float(x):
    """Format floats with 6 significant digits; otherwise return the value unchanged."""
    if isinstance(x, float):
        return format(x, ".6g")
    return x

In [7]:
# List of annotation row identifiers to exclude from averaging.
annotation_keys = ['EWEIGHT', 'NAME', 'GWEIGHT']

# Path to a single .pcl file for testing.
file_path = "/home/logs/jtorresb/yeastformer/yeast/yeast_data/data_inspection/dual_channel_extracted/Renaud-Young_2015_PMID_25701288/GSE66176.final.pcl"

try:
    # Read the file into a DataFrame (assuming tab-delimited and the first column is the index)
    df = pd.read_csv(file_path, sep="\t", header=0, index_col=0)

    # Separate annotation rows from the data rows.
    annotations = df.loc[df.index.isin(annotation_keys)]
    data = df.loc[~df.index.isin(annotation_keys)]

    # Build a dictionary grouping columns by their "base" name.
    groups = {}
    for col in data.columns:
        base = extract_base(col)
        groups.setdefault(base, []).append(col)

    # Identify if any averaging is needed.
    needs_averaging = any(len(cols) > 1 for cols in groups.values())

    if not needs_averaging:
        print("No replicates present. No averaging was performed.")
    else:
        # Create a new DataFrame for the averaged expression data.
        new_data = pd.DataFrame(index=data.index)
        for base, cols in groups.items():
            if len(cols) > 1:
                new_data[base] = data[cols].mean(axis=1)
            else:
                new_data[base] = data[cols[0]]
        
        # **Reindex annotations so that they use the new averaged column names:**
        annotations = annotations.reindex(columns=new_data.columns)

        # Combine the annotations with the averaged data.
        new_df = pd.concat([annotations, new_data])
    
        # Format all float values to 6 significant digits using Series.map.
        for col in new_df.columns:
            new_df[col] = new_df[col].map(format_float)
    
        # Overwrite the original file or write to a new file for testing.
        new_df.to_csv(file_path, sep="\t")
        print("File processed and written successfully.")
    
    # # For debugging/testing: Print the first few rows of the new dataframe.
    # print(new_df.head())

except Exception as e:
    print(f"Error processing {file_path}: {e}")

File processed and written successfully.


In [10]:
import os
import re
import pandas as pd
from tqdm import tqdm

# Annotation row identifiers to retain and handle specially.
annotation_keys = ['EWEIGHT', 'NAME', 'GWEIGHT']

def extract_base(col_name):
    pattern = re.compile(r"^(.*?)\s*(?:rep(?:licate)?|repeat)\s*[_-]?\s*\d+(.*)$", re.IGNORECASE)
    m = pattern.match(col_name)
    if m:
        base = (m.group(1) + m.group(2)).strip()
        return base if base else col_name
    return col_name

# Format function for floats.
def format_float(x):
    return format(x, ".6g") if isinstance(x, float) else x

# Root directory with subfolders containing .pcl files.
root_dir = "/home/logs/jtorresb/yeastformer/yeast/yeast_data/data_inspection/dual_channel_extracted"

# List all subfolders.
subfolders = [sub for sub in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, sub))]

# Process each .pcl file in all subfolders.
for subfolder in tqdm(subfolders, desc="Processing subfolders"):
    subfolder_path = os.path.join(root_dir, subfolder)
    pcl_files = [f for f in os.listdir(subfolder_path) if f.endswith(".pcl")]

    for file_name in pcl_files:
        file_path = os.path.join(subfolder_path, file_name)
        try:
            df = pd.read_csv(file_path, sep="\t", header=0, index_col=0)

            annotations = df.loc[df.index.isin(annotation_keys)]
            data = df.loc[~df.index.isin(annotation_keys)]

            groups = {}
            for col in data.columns:
                base = extract_base(col)
                groups.setdefault(base, []).append(col)

            if not any(len(cols) > 1 for cols in groups.values()):
                continue  # No averaging needed

            # Create averaged data.
            new_data = pd.DataFrame(index=data.index)
            for base, cols in groups.items():
                if len(cols) > 1:
                    new_data[base] = data[cols].mean(axis=1)
                else:
                    new_data[base] = data[cols[0]]

            # Create new annotation DataFrame with values from the first column of each group.
            new_annotations = pd.DataFrame(index=annotations.index, columns=new_data.columns)
            for base, cols in groups.items():
                first_col = cols[0]
                for key in annotation_keys:
                    if key in annotations.index:
                        new_annotations.at[key, base] = annotations.at[key, first_col]

            # Concatenate annotations + averaged expression data
            new_df = pd.concat([new_annotations, new_data])

            # Format floats
            new_df = new_df.applymap(format_float)

            # Save result
            new_df.to_csv(file_path, sep="\t")
        
        except Exception as e:
            print(f"Error processing {file_path}: {e}")

print("✅ Processing complete.")

Processing subfolders:   0%|          | 0/309 [00:00<?, ?it/s]/tmp/ipykernel_2375784/1260748054.py:65: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_df = pd.concat([new_annotations, new_data])
/tmp/ipykernel_2375784/1260748054.py:68: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  new_df = new_df.applymap(format_float)
Processing subfolders:  16%|█▌        | 49/309 [00:01<00:07, 32.84it/s]/tmp/ipykernel_2375784/1260748054.py:65: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the c

✅ Processing complete.


### Moving pcl files into a different folder

In [1]:
import os
import shutil

# Source root directory containing subfolders with .pcl files
root_dir = "/home/logs/jtorresb/yeastformer/yeast/yeast_data/data_inspection/dual_channel_extracted"

# Destination directory for all .pcl files
dest_dir = "/home/logs/jtorresb/yeastformer/yeast/yeast_data/dual_channel_pcls"
os.makedirs(dest_dir, exist_ok=True)

# Walk through all subdirectories in the root_dir
for dirpath, _, filenames in os.walk(root_dir):
    for file_name in filenames:
        if file_name.endswith(".pcl"):
            subfolder_name = os.path.basename(dirpath)
            new_file_name = f"{subfolder_name}_{file_name}"

            src_path = os.path.join(dirpath, file_name)
            dest_path = os.path.join(dest_dir, new_file_name)

            # Ensure the new file name is unique just in case
            i = 1
            base, ext = os.path.splitext(new_file_name)
            while os.path.exists(dest_path):
                dest_path = os.path.join(dest_dir, f"{base}_{i}{ext}")
                i += 1

            shutil.move(src_path, dest_path)
            # print(f"Moved: {src_path} → {dest_path}")

print("All .pcl files moved and renamed with subfolder prefix.")


All .pcl files moved and renamed with subfolder prefix.
